# Analytics: one view per capstone insight

This notebook builds the Analytics layer that the dashboard reads. It defines **one view per capstone insight** ([D-56](../../docs/decisions.md), proposed):

| # | Insight | View | One row per |
| --- | --- | --- | --- |
| 1 | Where does the money go? | `analytics_investment_distribution` | Geography level (region or province reporting area), geography key, and `infra_year` cohort |
| 2 | Are investments proportional to population? | `analytics_population_equity` | Province reporting area |
| 3 | Do flood-control investments overlap with flood-prone areas? | `analytics_flood_alignment` | Official PSGC region and flood point class |
| 4 | Which projects need a closer look? | `analytics_implementation_monitoring` | Current DPWH project |
| 5 | Are we seeing projects or a connected infrastructure network? | `analytics_infrastructure_network` | Current DPWH project |
| 6 | What can we see only after connecting the datasets? | `analytics_integration_coverage` | Integration link, link outcome, and region key |

The earlier eleven-view design (D-50, D-51, D-54), the proposed regional progress view (D-55), and the five `analytics_reporting_area_*` views first built for province reporting are retired. Section 0 drops their views, so the `05-analytics` schema holds only these views.

## Inputs

Gold only: `fact_project_snapshot`, `fact_region_population`, `fact_province_reporting_area_population`, `fact_region_flood_exposure`, `dim_project`, `dim_region`, `dim_province_reporting_area`, `dim_province_reporting_area_boundary`, `dim_project_status`, `dim_flood_susceptibility`, and `dim_date`. See the [Gold model](../../docs/model/gold_model.md). Views 2 and 3 also read view 1, so every geographic view uses the same members and coverage status.

## Two governed geographic levels

- **Region** is the official PSGC region in `fact_project_snapshot.region_key` ([D-31](../../docs/decisions.md)).
- **Province reporting area** is `fact_project_snapshot.reporting_area_key` ([D-57](../../docs/decisions.md)): an official PSGC province, a city with no province parent (`INDEPENDENT_CITY`), or NCR as one area. Silver assigned it once, mostly by the project point inside one province boundary. DPWH `location.province` never decides it.
- The two levels are assigned from different evidence, so a project can have a region but no reporting area, and the parent region of its reporting area can differ from its own region. Both keys stay as Gold publishes them. View 5 and view 6 show where they disagree.
- Never add the two levels together. Never derive region totals by adding reporting areas.

## Rules used in every view

- Reads only current Gold project rows (`is_current_snapshot = TRUE`), so each project counts once at each level.
- Never repeats Silver or Gold work. Region, reporting-area, and flood matching, category and status mapping, and the long-running rule come from Gold as published. No view parses geometry or runs a spatial join, and no view names a province.
- Budget means the reported project or contract budget, never payment ([D-28](../../docs/decisions.md)). A missing budget stays `NULL`. A reported `0` stays `0`.
- An unmapped or unresolved value stays `NULL` or keeps its own status row ([D-37](../../docs/decisions.md)). No view writes `Unknown` or `Unmapped` text.
- Key `0` holds Central Office, unmatched, ambiguous, and unresolved projects at each level. Views 1, 2, and 3 leave it out. Views 4, 5, and 6 keep it, and view 6 accounts for every key `0` project at both levels.
- `infra_year` is a **project cohort attribute** in the current snapshot. Gold holds one DPWH snapshot, so no view shows year-over-year portfolio history. Write "the 2018 infrastructure-year cohort in the current snapshot", never "DPWH investment in 2018".
- Every view carries `source_snapshot_id`, the DPWH snapshot behind its numbers ([D-29](../../docs/decisions.md)). View 3 also carries `mgb_source_snapshot_id`.
- Every view and column has a comment, so Genie can read what each one means.

In [0]:
%sql
-- every table is named with its schema: gold inputs as `03-gold`.table, the views as `05-analytics`.view
USE CATALOG `buildabida-capstone`;

## 0. Check the snapshot, create the schema, and retire the old views

### One current DPWH snapshot

**What:** stop unless the current rows of `fact_project_snapshot` hold exactly one DPWH snapshot, with no empty snapshot ID.

**Why:** every view reads `MAX(source_snapshot_id)` as the snapshot behind its numbers ([D-29](../../docs/decisions.md)). If current Gold mixed two snapshots, the views would add them together and still report one ID. `10_validation_analytics` checks the same rule again.

**Expected result:** the `ASSERT_TRUE` returns `NULL`, and no error is raised.

In [0]:
%sql
-- every view reads one dpwh snapshot, so stop before building anything if current gold mixes them
SELECT ASSERT_TRUE(
    COUNT(DISTINCT source_snapshot_id) = 1 AND COUNT_IF(source_snapshot_id IS NULL) = 0,
    'STOP: current Gold project rows span multiple DPWH snapshots. Rebuild Gold before analytics.'
)
FROM `03-gold`.fact_project_snapshot
WHERE is_current_snapshot = TRUE;

### Create the schema and drop the retired views

**What:** create `05-analytics` if it does not exist. Then drop every view of the earlier designs that this notebook no longer builds: the ten views of D-50, D-51, and D-54, the proposed `analytics_region_project_progress` view (D-55), and the five `analytics_reporting_area_*` views first built for D-57. `analytics_population_equity` keeps its name, so `CREATE OR REPLACE VIEW` replaces it below.

**Why:** `CREATE OR REPLACE VIEW` never removes a view that a notebook stops building. Without the drops, the dashboard and Genie could still read retired numbers. `DROP VIEW IF EXISTS` only touches views in `05-analytics` and holds no data, so a rerun is safe.

**Expected result:** the schema exists, and none of the retired views exists. A rerun changes nothing.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS `05-analytics`
COMMENT 'Analytics views for the dashboard and Genie. Built only from validated 03-gold tables.';

-- retired by d-56: the q1 to q6 and exploratory views of d-50, d-51, and d-54, the proposed d-55 view,
-- and the five reporting-area views first built for d-57
DROP VIEW IF EXISTS `05-analytics`.analytics_regional_investment;
DROP VIEW IF EXISTS `05-analytics`.analytics_category_portfolio;
DROP VIEW IF EXISTS `05-analytics`.analytics_project_delivery_status;
DROP VIEW IF EXISTS `05-analytics`.analytics_flood_control_exposure;
DROP VIEW IF EXISTS `05-analytics`.analytics_region_category;
DROP VIEW IF EXISTS `05-analytics`.analytics_infrastructure_year_trend;
DROP VIEW IF EXISTS `05-analytics`.analytics_infrastructure_year_sector;
DROP VIEW IF EXISTS `05-analytics`.analytics_delivery_cohort;
DROP VIEW IF EXISTS `05-analytics`.analytics_project_size_concentration;
DROP VIEW IF EXISTS `05-analytics`.analytics_implementing_office_portfolio;
DROP VIEW IF EXISTS `05-analytics`.analytics_region_project_progress;
DROP VIEW IF EXISTS `05-analytics`.analytics_reporting_area_investment;
DROP VIEW IF EXISTS `05-analytics`.analytics_reporting_area_category;
DROP VIEW IF EXISTS `05-analytics`.analytics_reporting_area_delivery_status;
DROP VIEW IF EXISTS `05-analytics`.analytics_reporting_area_population_equity;
DROP VIEW IF EXISTS `05-analytics`.analytics_reporting_area_project_size_concentration;

## 1. Where does the money go? (investment distribution)

**What:** build `analytics_investment_distribution`.

**Business question:** which regions and provinces account for the largest share of recorded infrastructure investment, and is that investment concentrated in a few areas or broadly distributed?

**Grain:** one row per `geography_level`, `geography_key`, and `infra_year` cohort.

- `geography_level = 'REGION'`: one official PSGC region, from `fact_project_snapshot.region_key`.
- `geography_level = 'REPORTING_AREA'`: one province reporting area, from `fact_project_snapshot.reporting_area_key` (D-57). `geography_type` says whether it is a `PROVINCE`, an `INDEPENDENT_CITY`, or `NCR`.
- A project without an `infra_year` keeps its own `NULL` cohort row. A member with no mapped project keeps one row with `project_count` 0 and `cohort_row_type` `NO_MAPPED_PROJECT`.
- Key `0` is left out at both levels. `analytics_integration_coverage` accounts for it.

Every project appears **once at each level**. Always filter one level. Never add the two levels together.

**Why:** a unified grain lets one dashboard dataset switch between regions and provinces with the same columns. Region rows keep the region assignment exactly as before, so region totals are not rebuilt from provinces. That matters because about a quarter of projects have a region but no reporting area, and the parent region of a reporting area can differ from the project region.

- The additive measures (`project_count`, `projects_with_budget`, `total_reported_budget_pesos`) sit at the level, member, and cohort grain, so a filtered chart can add them up within one level.
- `cohort_*` columns compare members **within one level and one cohort**. Use them when the cohort filter selects exactly one year.
- `geography_*` columns describe the member **over all cohorts**. They repeat on every cohort row of the member. Read them with `MAX`, never `SUM`, in a dataset that the cohort filter does not touch.
- `source_coverage_status` is `REPORTED`, `LIMITED_SOURCE_COVERAGE` for BARMM and its reporting areas when they have mapped projects, or `NO_MAPPED_PROJECT_DATA`. Only `REPORTED` members are ranked.
- `boundary_status` comes from the Gold reporting-area boundary dimension. A reporting area without a valid boundary, such as an independent city, cannot receive projects by coordinates, so its zero is a coverage gap, not low investment.
- Shares divide by the budget of every member of the same level. `*_cumulative_budget_share_pct` adds the shares of the ranked members from rank 1 down, so at rank 3 it reads "the top three hold this percent".

**Interpretation rule:** these numbers describe where reported budget sits. They do not show need, fairness, or adequacy. A cohort is a group of projects as they look in the current snapshot, not the DPWH portfolio in that year.

**Expected result:** within each level, the rows add up to every current Gold project and budget outside key `0`, and within each level and cohort, `cohort_budget_share_pct` adds up to 100.

In [0]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_investment_distribution (
    geography_level COMMENT 'REGION for an official PSGC region or REPORTING_AREA for a D-57 province reporting area. Always filter one level. Never add the two levels together.',
    geography_key COMMENT 'region_key from dim_region when the level is REGION, or reporting_area_key from dim_province_reporting_area when the level is REPORTING_AREA. Key 0 is never in this view.',
    psgc_code COMMENT 'Official PSGC code of the region or reporting area.',
    geography_name COMMENT 'Official PSGC name of the region or reporting area.',
    geography_short_name COMMENT 'Short label for charts.',
    geography_type COMMENT 'REGION, PROVINCE, INDEPENDENT_CITY, or NCR, from Gold.',
    parent_region_key COMMENT 'Region of the member. The region itself on a REGION row, or the PSGC parent region of a reporting area.',
    parent_region_name COMMENT 'Official name of parent_region_key.',
    centroid_latitude COMMENT 'Boundary centroid latitude from Gold, for a map. NULL when Gold has no safe boundary.',
    centroid_longitude COMMENT 'Boundary centroid longitude from Gold, for a map. NULL when Gold has no safe boundary.',
    boundary_status COMMENT 'Newest Gold boundary status of a reporting area, such as VALID_BOUNDARY or MISSING_BOUNDARY. A reporting area without a valid boundary cannot receive projects by coordinates. NULL on REGION rows.',
    source_coverage_status COMMENT 'Member coverage over all cohorts. REPORTED, LIMITED_SOURCE_COVERAGE for BARMM and its reporting areas with mapped projects, or NO_MAPPED_PROJECT_DATA. Only REPORTED members are ranked.',
    infra_year COMMENT 'Infrastructure-year cohort from dim_project.infra_year. A project cohort in the current snapshot, not a past state of the DPWH portfolio. NULL when the source value is not a whole year, and on the row of a member with no mapped project.',
    cohort_row_type COMMENT 'INFRA_YEAR for a cohort with a year, NO_INFRA_YEAR for projects whose infra_year is NULL, or NO_MAPPED_PROJECT for the one row of a member with no mapped project.',
    project_count COMMENT 'Distinct current DPWH projects of the member and cohort. Additive within one level.',
    projects_with_budget COMMENT 'Projects of the member and cohort with a reported budget that is not NULL. Additive within one level.',
    total_reported_budget_pesos COMMENT 'Sum of reported project or contract budgets in pesos. Not actual payment. NULL when no project has a budget, never 0 for a missing value. Additive within one level.',
    avg_budget_per_project_pesos COMMENT 'total_reported_budget_pesos divided by projects_with_budget. NULL when no project has a budget.',
    cohort_budget_share_pct COMMENT 'Member budget as a percent of the budget of every member of the same level in the same cohort. Valid when one level and one cohort are selected.',
    cohort_budget_rank COMMENT '1 means the highest reported budget within the level and cohort. REPORTED members only. Ties share a rank.',
    cohort_project_count_rank COMMENT '1 means the most projects within the level and cohort. REPORTED members only. Ties share a rank.',
    cohort_cumulative_budget_share_pct COMMENT 'Running total of cohort_budget_share_pct from rank 1 down, over REPORTED members of the level. At rank 3 it is the share held by the top three. NULL when the row has no reported budget.',
    geography_project_count COMMENT 'Distinct current projects of the member over all cohorts. Repeats on every cohort row. Read with MAX, never SUM.',
    geography_budget_pesos COMMENT 'Reported budget of the member over all cohorts. Repeats on every cohort row. Read with MAX, never SUM. NULL when no project has a budget.',
    geography_budget_share_pct COMMENT 'geography_budget_pesos as a percent of the budget of every member of the same level over all cohorts. Repeats on every cohort row.',
    geography_budget_rank COMMENT '1 means the highest reported budget in the level over all cohorts. REPORTED members only. Ties share a rank. Repeats on every cohort row.',
    geography_project_count_rank COMMENT '1 means the most projects in the level over all cohorts. REPORTED members only. Ties share a rank. Repeats on every cohort row.',
    geography_cumulative_budget_share_pct COMMENT 'Running total of geography_budget_share_pct from rank 1 down, over REPORTED members of the level. At rank 3 it is the share held by the top three. NULL when the member has no reported budget. Repeats on every cohort row.',
    snapshot_date COMMENT 'File date of the DPWH snapshot behind every number, from dim_date. The data is as of this date.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).'
)
COMMENT 'Insight 1, investment distribution. One row per geography level, geography key, and infra_year cohort with project count, reported budget, budget share, ranks, and cumulative concentration share within the cohort and over all cohorts. The levels are REGION and REPORTING_AREA (D-57 province reporting area). Every project appears once at each level, so always filter one level. infra_year is a project cohort in the current snapshot, not a historical snapshot of the DPWH portfolio. Reported budget is not payment. Members with no or limited DPWH coverage stay visible and are not ranked. Key 0 is left out.'
AS
WITH current_projects AS (
    -- one current row per project. 10_validation_analytics stops the run if that breaks
    SELECT
        fact.project_key,
        fact.region_key,
        fact.reporting_area_key,
        fact.reported_budget_pesos,
        project.infra_year
    FROM `03-gold`.fact_project_snapshot AS fact
    -- left join keeps every current project (d-36). a null year keeps its own cohort row
    LEFT JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    WHERE fact.is_current_snapshot = TRUE
),

project_geography AS (
    -- each project once per level, from its governed gold key. the two levels are never added together
    SELECT
        'REGION' AS geography_level,
        region_key AS geography_key,
        project_key,
        reported_budget_pesos,
        infra_year
    FROM current_projects
    WHERE region_key <> 0
    UNION ALL
    SELECT
        'REPORTING_AREA' AS geography_level,
        reporting_area_key AS geography_key,
        project_key,
        reported_budget_pesos,
        infra_year
    FROM current_projects
    WHERE reporting_area_key <> 0
),

current_snapshot AS (
    SELECT
        MAX(fact.source_snapshot_id) AS source_snapshot_id,
        MAX(snapshot_day.calendar_date) AS snapshot_date
    FROM `03-gold`.fact_project_snapshot AS fact
    LEFT JOIN `03-gold`.dim_date AS snapshot_day
        ON fact.snapshot_date_key = snapshot_day.date_key
    WHERE fact.is_current_snapshot = TRUE
),

region_versions AS (
    -- the psgc version behind the current region keys
    SELECT DISTINCT region.psgc_version
    FROM current_projects AS project
    INNER JOIN `03-gold`.dim_region AS region
        ON project.region_key = region.region_key
    WHERE project.region_key <> 0
),

area_versions AS (
    -- the psgc version behind the current reporting-area keys
    SELECT DISTINCT area.psgc_version
    FROM current_projects AS project
    INNER JOIN `03-gold`.dim_province_reporting_area AS area
        ON project.reporting_area_key = area.reporting_area_key
    WHERE project.reporting_area_key <> 0
),

area_boundaries AS (
    -- each reporting area's boundary rows, newest first
    SELECT
        reporting_area_key,
        boundary_status,
        ROW_NUMBER() OVER (
            PARTITION BY reporting_area_key
            ORDER BY source_load_ts DESC, boundary_version DESC
        ) AS boundary_rank
    FROM `03-gold`.dim_province_reporting_area_boundary
),

members AS (
    -- every official member of each level keeps a row, even one with no mapped project
    SELECT
        'REGION' AS geography_level,
        region.region_key AS geography_key,
        region.psgc_region_code AS psgc_code,
        region.region_name AS geography_name,
        region.region_short_name AS geography_short_name,
        'REGION' AS geography_type,
        region.region_key AS parent_region_key,
        region.region_name AS parent_region_name,
        region.centroid_latitude,
        region.centroid_longitude,
        CAST(NULL AS STRING) AS boundary_status,
        region.is_barmm
    FROM `03-gold`.dim_region AS region
    INNER JOIN region_versions AS version
        ON region.psgc_version <=> version.psgc_version
    WHERE region.region_key <> 0
    UNION ALL
    SELECT
        'REPORTING_AREA' AS geography_level,
        area.reporting_area_key AS geography_key,
        area.psgc_reporting_area_code AS psgc_code,
        area.reporting_area_name AS geography_name,
        area.reporting_area_short_name AS geography_short_name,
        area.reporting_area_type AS geography_type,
        area.region_key AS parent_region_key,
        parent.region_name AS parent_region_name,
        area.centroid_latitude,
        area.centroid_longitude,
        boundary.boundary_status,
        COALESCE(parent.is_barmm, FALSE) AS is_barmm
    FROM `03-gold`.dim_province_reporting_area AS area
    INNER JOIN area_versions AS version
        ON area.psgc_version <=> version.psgc_version
    LEFT JOIN `03-gold`.dim_region AS parent
        ON area.region_key = parent.region_key
    LEFT JOIN area_boundaries AS boundary
        ON area.reporting_area_key = boundary.reporting_area_key
        AND boundary.boundary_rank = 1
    WHERE area.reporting_area_key <> 0
),

cohort_totals AS (
    SELECT
        geography_level,
        geography_key,
        infra_year,
        COUNT(DISTINCT project_key) AS project_count,
        COUNT(DISTINCT CASE WHEN reported_budget_pesos IS NOT NULL THEN project_key END) AS projects_with_budget,
        SUM(reported_budget_pesos) AS total_reported_budget_pesos
    FROM project_geography
    GROUP BY ALL
),

geography_totals AS (
    SELECT
        geography_level,
        geography_key,
        SUM(project_count) AS geography_project_count,
        SUM(total_reported_budget_pesos) AS geography_budget_pesos
    FROM cohort_totals
    GROUP BY ALL
),

geographies AS (
    SELECT
        member.geography_level,
        member.geography_key,
        member.psgc_code,
        member.geography_name,
        member.geography_short_name,
        member.geography_type,
        member.parent_region_key,
        member.parent_region_name,
        member.centroid_latitude,
        member.centroid_longitude,
        member.boundary_status,
        COALESCE(totals.geography_project_count, 0) AS geography_project_count,
        -- no coalesce. a member without a budgeted project keeps a null budget, never 0
        totals.geography_budget_pesos,
        CASE
            WHEN COALESCE(totals.geography_project_count, 0) = 0 THEN 'NO_MAPPED_PROJECT_DATA'
            -- readme and d-28: dpwh coverage of barmm is limited by the source
            WHEN member.is_barmm THEN 'LIMITED_SOURCE_COVERAGE'
            ELSE 'REPORTED'
        END AS source_coverage_status
    FROM members AS member
    LEFT JOIN geography_totals AS totals
        ON member.geography_level = totals.geography_level
        AND member.geography_key = totals.geography_key
),

ranked_geographies AS (
    -- each level and status is ranked on its own, and only reported ranks are published
    SELECT
        geographies.*,
        ROUND(
            100.0 * geography_budget_pesos
            / NULLIF(SUM(geography_budget_pesos) OVER (PARTITION BY geography_level), 0),
            4
        ) AS geography_budget_share_pct,
        RANK() OVER (
            PARTITION BY geography_level, source_coverage_status
            ORDER BY geography_budget_pesos DESC NULLS LAST
        ) AS geography_budget_rank_in_status,
        RANK() OVER (
            PARTITION BY geography_level, source_coverage_status
            ORDER BY geography_project_count DESC
        ) AS geography_project_count_rank_in_status,
        SUM(geography_budget_pesos) OVER (
            PARTITION BY geography_level, source_coverage_status
            ORDER BY geography_budget_pesos DESC NULLS LAST, geography_key
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS geography_running_budget_in_status,
        SUM(geography_budget_pesos) OVER (PARTITION BY geography_level) AS level_budget_pesos
    FROM geographies
),

cohort_rows AS (
    -- every member from above, so a member with no mapped project keeps one row
    SELECT
        geography.*,
        cohort.infra_year,
        CASE
            WHEN cohort.geography_key IS NULL THEN 'NO_MAPPED_PROJECT'
            WHEN cohort.infra_year IS NULL THEN 'NO_INFRA_YEAR'
            ELSE 'INFRA_YEAR'
        END AS cohort_row_type,
        COALESCE(cohort.project_count, 0) AS project_count,
        COALESCE(cohort.projects_with_budget, 0) AS projects_with_budget,
        cohort.total_reported_budget_pesos
    FROM ranked_geographies AS geography
    LEFT JOIN cohort_totals AS cohort
        ON geography.geography_level = cohort.geography_level
        AND geography.geography_key = cohort.geography_key
),

ranked_cohorts AS (
    -- the same ranking rules, within one level and cohort. a null year is its own cohort
    SELECT
        cohort_rows.*,
        RANK() OVER (
            PARTITION BY geography_level, infra_year, source_coverage_status
            ORDER BY total_reported_budget_pesos DESC NULLS LAST
        ) AS cohort_budget_rank_in_status,
        RANK() OVER (
            PARTITION BY geography_level, infra_year, source_coverage_status
            ORDER BY project_count DESC
        ) AS cohort_project_count_rank_in_status,
        SUM(total_reported_budget_pesos) OVER (
            PARTITION BY geography_level, infra_year, source_coverage_status
            ORDER BY total_reported_budget_pesos DESC NULLS LAST, geography_key
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS cohort_running_budget_in_status,
        SUM(total_reported_budget_pesos) OVER (PARTITION BY geography_level, infra_year) AS cohort_budget_pesos
    FROM cohort_rows
)

SELECT
    ranked_cohorts.geography_level,
    ranked_cohorts.geography_key,
    ranked_cohorts.psgc_code,
    ranked_cohorts.geography_name,
    ranked_cohorts.geography_short_name,
    ranked_cohorts.geography_type,
    ranked_cohorts.parent_region_key,
    ranked_cohorts.parent_region_name,
    ranked_cohorts.centroid_latitude,
    ranked_cohorts.centroid_longitude,
    ranked_cohorts.boundary_status,
    ranked_cohorts.source_coverage_status,
    ranked_cohorts.infra_year,
    ranked_cohorts.cohort_row_type,
    ranked_cohorts.project_count,
    ranked_cohorts.projects_with_budget,
    ranked_cohorts.total_reported_budget_pesos,
    ROUND(
        ranked_cohorts.total_reported_budget_pesos / NULLIF(ranked_cohorts.projects_with_budget, 0), 2
    ) AS avg_budget_per_project_pesos,
    ROUND(
        100.0 * ranked_cohorts.total_reported_budget_pesos / NULLIF(ranked_cohorts.cohort_budget_pesos, 0), 4
    ) AS cohort_budget_share_pct,
    CASE
        WHEN ranked_cohorts.source_coverage_status = 'REPORTED' THEN ranked_cohorts.cohort_budget_rank_in_status
    END AS cohort_budget_rank,
    CASE
        WHEN ranked_cohorts.source_coverage_status = 'REPORTED'
            THEN ranked_cohorts.cohort_project_count_rank_in_status
    END AS cohort_project_count_rank,
    -- a member without a reported budget adds nothing, so it gets no running share
    CASE
        WHEN ranked_cohorts.source_coverage_status = 'REPORTED'
            AND ranked_cohorts.total_reported_budget_pesos IS NOT NULL THEN ROUND(
            100.0 * ranked_cohorts.cohort_running_budget_in_status / NULLIF(ranked_cohorts.cohort_budget_pesos, 0), 4
        )
    END AS cohort_cumulative_budget_share_pct,
    ranked_cohorts.geography_project_count,
    ranked_cohorts.geography_budget_pesos,
    ranked_cohorts.geography_budget_share_pct,
    CASE
        WHEN ranked_cohorts.source_coverage_status = 'REPORTED'
            THEN ranked_cohorts.geography_budget_rank_in_status
    END AS geography_budget_rank,
    CASE
        WHEN ranked_cohorts.source_coverage_status = 'REPORTED'
            THEN ranked_cohorts.geography_project_count_rank_in_status
    END AS geography_project_count_rank,
    CASE
        WHEN ranked_cohorts.source_coverage_status = 'REPORTED'
            AND ranked_cohorts.geography_budget_pesos IS NOT NULL THEN ROUND(
            100.0 * ranked_cohorts.geography_running_budget_in_status
            / NULLIF(ranked_cohorts.level_budget_pesos, 0),
            4
        )
    END AS geography_cumulative_budget_share_pct,
    current_snapshot.snapshot_date,
    current_snapshot.source_snapshot_id
FROM ranked_cohorts
CROSS JOIN current_snapshot
ORDER BY
    ranked_cohorts.geography_level,
    ranked_cohorts.infra_year NULLS LAST,
    cohort_budget_rank NULLS LAST,
    ranked_cohorts.geography_name;

## 2. Are investments proportional to population? (population equity)

**What:** build `analytics_population_equity`.

**Business question:** which provinces receive a larger or smaller share of recorded investment than their share of the national population?

**Grain:** one row per province reporting area (D-57): an official PSGC province, an independent city, or NCR as one area. Key `0` is left out, because unresolved projects and unresolved population have no area (#61).

**Why:**

- Area, parent region, project count, budget, and coverage come from the `REPORTING_AREA` rows of `analytics_investment_distribution` over all cohorts, so views 1 and 2 always agree. The view reads the member-level columns with `SELECT DISTINCT`, and the validator proves one row per area.
- Population comes from `fact_province_reporting_area_population` where `is_primary_source = TRUE`: the same accepted Census Table C barangay rows that feed the region totals (D-18, D-24, D-57). Only the newest primary year is read. No other population source is used.
- Both shares in the gap use the same comparable areas: `REPORTED`, with a budget, and with a population. `equity_comparison_status` says why an area is outside that set. An area outside it keeps its population and population share, but its investment share, gap, ratio, and per-capita figures stay `NULL`. An area without a valid boundary or with limited source coverage, such as a BARMM province, would otherwise read as low investment.
- "National included" means the comparable areas. `share_gap_pp` is the investment share minus the comparable population share, in percentage points. `investment_to_population_share_ratio` divides the two, so 1 means they match.
- The view has no "substantially different" flag. A threshold needs its own decision. The dashboard sorts by the continuous gap instead.

**Interpretation rule:** this is a **screening indicator, not a verdict on fairness**. A gap can reflect geography, existing infrastructure gaps, construction costs, who benefits from a project, project type mix, or source coverage. About a quarter of projects have no reporting area, so the comparison covers mapped projects only. Population is one proxy for need.

**Expected result:** one row per geographic reporting area. Investment shares and comparable population shares each add up to 100 across comparable areas. Population shares add up to 100 across every area with a population.

In [0]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_population_equity (
    reporting_area_key COMMENT 'Province reporting area (D-57). Joins to dim_province_reporting_area.reporting_area_key. Key 0 is never in this view.',
    psgc_reporting_area_code COMMENT 'Official PSGC code of the province, independent city, or NCR.',
    reporting_area_name COMMENT 'Official PSGC name of the reporting area.',
    reporting_area_short_name COMMENT 'Short label for charts.',
    reporting_area_type COMMENT 'PROVINCE, INDEPENDENT_CITY, or NCR, from Gold.',
    region_key COMMENT 'PSGC parent region of the reporting area. Joins to dim_region.region_key.',
    region_name COMMENT 'Official name of the parent region.',
    boundary_status COMMENT 'Newest Gold boundary status of the area. An area without a valid boundary cannot receive projects by coordinates.',
    source_coverage_status COMMENT 'Coverage status from analytics_investment_distribution. REPORTED, LIMITED_SOURCE_COVERAGE, or NO_MAPPED_PROJECT_DATA.',
    equity_comparison_status COMMENT 'COMPARABLE, or why the area is outside the comparison: EXCLUDED_SOURCE_COVERAGE, EXCLUDED_NO_REPORTED_BUDGET, or EXCLUDED_NO_POPULATION.',
    project_count COMMENT 'Distinct current DPWH projects mapped to the area over all cohorts.',
    total_reported_budget_pesos COMMENT 'Reported budget of the area over all cohorts. Not actual payment. NULL when no project has a budget.',
    population_reference_year COMMENT 'Census year of the primary population source.',
    population_count COMMENT 'Primary-source population of the area from fact_province_reporting_area_population. NULL when Gold has none.',
    population_source_name COMMENT 'Name of the primary population source.',
    population_share_pct COMMENT 'Area population as a percent of the population of every reporting area with a population.',
    comparable_population_share_pct COMMENT 'Area population as a percent of the population of the comparable areas, the national included population. NULL when not comparable.',
    investment_share_pct COMMENT 'Area budget as a percent of the budget of the comparable areas, the national included investment. NULL when not comparable.',
    share_gap_pp COMMENT 'investment_share_pct minus comparable_population_share_pct, in percentage points. Above 0 means a larger investment share than population share. A screening indicator, not a fairness verdict. NULL when not comparable.',
    investment_to_population_share_ratio COMMENT 'investment_share_pct divided by comparable_population_share_pct. 1 means the shares match. NULL when not comparable.',
    investment_per_capita_pesos COMMENT 'Reported budget divided by population, in pesos per person. NULL when not comparable.',
    investment_per_capita_rank COMMENT '1 means the highest reported budget per person among comparable areas. Ties share a rank. NULL when not comparable.',
    comparable_set_per_capita_pesos COMMENT 'Reported budget of all comparable areas divided by their population. A reference line for the scatterplot. NULL when not comparable.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).',
    population_source_snapshot_id COMMENT 'Census Table C snapshot behind the population (D-29).'
)
COMMENT 'Insight 2, population equity. One row per province reporting area (D-57) with reported budget, primary census population, investment and population shares over the same comparable areas, the share gap, the share ratio, and investment per capita. A screening indicator, not a fairness verdict. Key 0 is left out.'
AS
WITH areas AS (
    -- the member-level columns repeat on every cohort row of view 1, so distinct gives one row per area
    SELECT DISTINCT
        geography_key,
        psgc_code,
        geography_name,
        geography_short_name,
        geography_type,
        parent_region_key,
        parent_region_name,
        boundary_status,
        source_coverage_status,
        geography_project_count,
        geography_budget_pesos,
        source_snapshot_id
    FROM `05-analytics`.analytics_investment_distribution
    WHERE geography_level = 'REPORTING_AREA'
),

primary_population AS (
    SELECT
        reporting_area_key,
        reference_year,
        population_count,
        source_name,
        source_snapshot_id
    FROM `03-gold`.fact_province_reporting_area_population
    WHERE is_primary_source = TRUE
        AND reporting_area_key <> 0
    -- the newest primary census year only, so two years never add up
    QUALIFY reference_year = MAX(reference_year) OVER ()
),

combined AS (
    SELECT
        area.geography_key AS reporting_area_key,
        area.psgc_code AS psgc_reporting_area_code,
        area.geography_name AS reporting_area_name,
        area.geography_short_name AS reporting_area_short_name,
        area.geography_type AS reporting_area_type,
        area.parent_region_key AS region_key,
        area.parent_region_name AS region_name,
        area.boundary_status,
        area.source_coverage_status,
        area.geography_project_count AS project_count,
        area.geography_budget_pesos AS total_reported_budget_pesos,
        population.reference_year AS population_reference_year,
        population.population_count,
        population.source_name AS population_source_name,
        population.source_snapshot_id AS population_source_snapshot_id,
        -- one comparable set for both shares: reported coverage, a budget, and a population
        CASE
            WHEN area.source_coverage_status <> 'REPORTED' THEN 'EXCLUDED_SOURCE_COVERAGE'
            WHEN area.geography_budget_pesos IS NULL THEN 'EXCLUDED_NO_REPORTED_BUDGET'
            WHEN population.population_count IS NULL THEN 'EXCLUDED_NO_POPULATION'
            ELSE 'COMPARABLE'
        END AS equity_comparison_status,
        area.source_snapshot_id
    FROM areas AS area
    LEFT JOIN primary_population AS population
        ON area.geography_key = population.reporting_area_key
),

comparable AS (
    SELECT
        combined.*,
        CASE WHEN equity_comparison_status = 'COMPARABLE' THEN total_reported_budget_pesos END AS comparable_budget_pesos,
        CASE WHEN equity_comparison_status = 'COMPARABLE' THEN population_count END AS comparable_population
    FROM combined
),

shares AS (
    SELECT
        comparable.*,
        ROUND(100.0 * population_count / NULLIF(SUM(population_count) OVER (), 0), 4) AS population_share_pct,
        ROUND(
            100.0 * comparable_population / NULLIF(SUM(comparable_population) OVER (), 0), 4
        ) AS comparable_population_share_pct,
        ROUND(
            100.0 * comparable_budget_pesos / NULLIF(SUM(comparable_budget_pesos) OVER (), 0), 4
        ) AS investment_share_pct,
        ROUND(comparable_budget_pesos / NULLIF(comparable_population, 0), 2) AS investment_per_capita_pesos,
        ROUND(
            SUM(comparable_budget_pesos) OVER () / NULLIF(SUM(comparable_population) OVER (), 0), 2
        ) AS all_comparable_per_capita_pesos
    FROM comparable
)

SELECT
    reporting_area_key,
    psgc_reporting_area_code,
    reporting_area_name,
    reporting_area_short_name,
    reporting_area_type,
    region_key,
    region_name,
    boundary_status,
    source_coverage_status,
    equity_comparison_status,
    project_count,
    total_reported_budget_pesos,
    population_reference_year,
    population_count,
    population_source_name,
    population_share_pct,
    comparable_population_share_pct,
    investment_share_pct,
    investment_share_pct - comparable_population_share_pct AS share_gap_pp,
    ROUND(investment_share_pct / NULLIF(comparable_population_share_pct, 0), 4) AS investment_to_population_share_ratio,
    investment_per_capita_pesos,
    CASE
        WHEN equity_comparison_status = 'COMPARABLE' THEN RANK() OVER (
            PARTITION BY equity_comparison_status
            ORDER BY investment_per_capita_pesos DESC
        )
    END AS investment_per_capita_rank,
    CASE WHEN equity_comparison_status = 'COMPARABLE' THEN all_comparable_per_capita_pesos END
        AS comparable_set_per_capita_pesos,
    source_snapshot_id,
    population_source_snapshot_id
FROM shares
ORDER BY share_gap_pp DESC NULLS LAST, reporting_area_name;

## 3. Do flood-control investments overlap with flood-prone areas? (flood alignment)

**What:** build `analytics_flood_alignment`.

**Business question:** where are flood-control investments located relative to mapped MGB flood susceptibility, and how much recorded investment and how many projects fall in each susceptibility level?

**Grain:** one row per official PSGC region and flood point class. Every region has one row for each class, even with no project, so the map legend and the area figures stay complete. Key `0` is left out. The regions are the `REGION` rows of view 1.

The flood point class is the Gold flood result at the project's own map point:

| Class | From Gold |
| --- | --- |
| `Low`, `Moderate`, `High`, `Very High` | `flood_match_status = 'MATCHED'`, the level from `dim_flood_susceptibility` |
| `AMBIGUOUS` | The point sits in polygons of more than one level. No level is picked ([D-45](../../docs/decisions.md)). |
| `OUTSIDE_MAPPED_ZONES` | `UNMATCHED` with a valid point. MGB has no polygon there. Not low risk. |
| `NO_USABLE_POINT` | `UNMATCHED` without a valid point. Classification is unavailable. |
| `INVALID_SOURCE` | Silver could not classify the source row. |

**Why:**

- The spatial match ran once in Silver Table 15. Gold resolved it to a key. This view only joins and counts. It parses no geometry and runs no spatial join.
- Flood-control is the broad DPWH flag `is_dpwh_flood_related` ([D-39](../../docs/decisions.md)), which includes the whole budget of Multi-sector projects with a flood component. The strict single-sector figures and the official flood-list figures sit beside it, so the real amount can be bracketed.
- Every project in a row is flood-control, other, or uncategorized. The three counts add up to `project_count`, so no project is hidden.
- The area columns come from the newest Gold exposure run. They fill only the level rows. Levels can overlap, so add level areas within a region only when `exposure_area_status` is `ADDITIVE_WITHIN_TOLERANCE` ([Gold model](../../docs/model/gold_model.md#fact_region_flood_exposure)).
- A region with no mapped DPWH project, such as BARMM, keeps `NULL` project figures, because the source cannot say.

**Interpretation rule:** overlap does not prove that a project protects an area, or that it protects a whole region. A point outside mapped MGB polygons is **not** low risk. Keep `AMBIGUOUS`, `OUTSIDE_MAPPED_ZONES`, `NO_USABLE_POINT`, and `INVALID_SOURCE` visible next to the level rows.

**Why region and not province:** the MGB area of each flood level exists only per region in Gold. Province flood exposure is deferred until its one-time Silver intersection is proven safe and affordable (D-57). Province flood context is available per project in view 5, from the same Gold flood result.

**Expected result:** one row per region and class. The class rows of each mapped region add up to its region project count in view 1.

In [0]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_flood_alignment (
    region_key COMMENT 'Official region. Joins to dim_region.region_key. Key 0 is never in this view.',
    region_name COMMENT 'Official PSGC region name.',
    region_short_name COMMENT 'Short region label for charts.',
    source_coverage_status COMMENT 'Coverage status from analytics_investment_distribution. NO_MAPPED_PROJECT_DATA means the project figures are unknown, not zero.',
    flood_point_class COMMENT 'Gold flood result at the project map point: an MGB level (Low, Moderate, High, Very High), AMBIGUOUS, OUTSIDE_MAPPED_ZONES, NO_USABLE_POINT, or INVALID_SOURCE. OUTSIDE_MAPPED_ZONES is not low risk.',
    is_mgb_level COMMENT 'True when flood_point_class is an approved MGB level. False for the classes that have no single level.',
    severity_rank COMMENT 'MGB severity rank of the level, for sorting. NULL for the classes that have no single level.',
    project_count COMMENT 'Distinct current DPWH projects in the region whose point falls in this class. NULL when no DPWH project maps to the region.',
    total_reported_budget_pesos COMMENT 'Reported budget of those projects in pesos. Not actual payment. NULL when none has a budget or no DPWH project maps to the region.',
    flood_control_project_count COMMENT 'Projects with a Flood Control and Drainage component, the broad D-39 flag, including Multi-sector projects. NULL when no DPWH project maps to the region.',
    flood_control_budget_pesos COMMENT 'Reported budget of those flood-control projects, including the whole budget of Multi-sector projects. NULL when none has a budget.',
    flood_control_single_sector_project_count COMMENT 'Projects whose only sector is Flood Control and Drainage, the strict flag. NULL when no DPWH project maps to the region.',
    flood_control_single_sector_budget_pesos COMMENT 'Reported budget of the strict single-sector flood-control projects. NULL when none has a budget.',
    official_flood_list_project_count COMMENT 'Projects whose Contract ID matches the official flood-control list (dim_project.is_in_official_flood_list). NULL when no DPWH project maps to the region.',
    official_flood_list_budget_pesos COMMENT 'DPWH reported budget of the official flood-list projects. Flood-list cost is never added. NULL when none has a budget.',
    other_project_count COMMENT 'Projects with approved categories and no flood component. NULL when no DPWH project maps to the region.',
    uncategorized_project_count COMMENT 'Projects with no usable category, so the flood flag is NULL (D-37). flood_control, other, and uncategorized add up to project_count.',
    share_of_region_flood_control_projects_pct COMMENT 'flood_control_project_count as a percent of the flood-control projects of the region. The classes of one region add up to 100.',
    share_of_region_flood_control_budget_pct COMMENT 'flood_control_budget_pesos as a percent of the flood-control budget of the region. The classes of one region add up to 100.',
    susceptible_area_sqkm COMMENT 'Region area in this MGB level, in square kilometers, from the newest Gold exposure run. NULL on rows that are not a level and where Gold has no exposure row.',
    share_of_region_area_pct COMMENT 'Share of the region area in this MGB level, from Gold. NULL on rows that are not a level and where Gold has no exposure row.',
    exposure_area_status COMMENT 'Region exposure status: ADDITIVE_WITHIN_TOLERANCE, NON_ADDITIVE_CROSS_LEVEL_OVERLAP, NOT_EVALUATED when no safe boundary exists, or NO_EXPOSURE_DATA. Add level areas only when ADDITIVE_WITHIN_TOLERANCE.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind the project numbers (D-29).',
    mgb_source_snapshot_id COMMENT 'MGB snapshot behind the area numbers (D-29).'
)
COMMENT 'Insight 3, flood alignment. One row per official PSGC region and flood point class with all, flood-control, strict flood-control, and official flood-list project counts and reported budgets, and the MGB area of each level. Reuses the Silver spatial match through Gold. Overlap does not prove protection, and a point outside mapped zones is not low risk. Key 0 is left out.'
AS
WITH regions AS (
    -- the region rows of view 1, one row per region
    SELECT DISTINCT
        geography_key AS region_key,
        geography_name AS region_name,
        geography_short_name AS region_short_name,
        source_coverage_status,
        source_snapshot_id
    FROM `05-analytics`.analytics_investment_distribution
    WHERE geography_level = 'REGION'
),

point_classes AS (
    -- the approved mgb levels, then the classes that have no single level
    SELECT
        flood_susceptibility_level AS flood_point_class,
        TRUE AS is_mgb_level,
        MAX(severity_rank) AS severity_rank
    FROM `03-gold`.dim_flood_susceptibility
    WHERE flood_susceptibility_key <> 0
    GROUP BY ALL
    UNION ALL
    SELECT
        unclassified.flood_point_class,
        FALSE AS is_mgb_level,
        CAST(NULL AS INT) AS severity_rank
    FROM VALUES ('AMBIGUOUS'), ('OUTSIDE_MAPPED_ZONES'), ('NO_USABLE_POINT'), ('INVALID_SOURCE')
        AS unclassified (flood_point_class)
),

project_points AS (
    SELECT
        fact.region_key,
        fact.project_key,
        fact.reported_budget_pesos,
        project.is_dpwh_flood_related,
        project.standardized_sector,
        project.is_in_official_flood_list,
        -- the gold result of the silver table 15 point match. no spatial work here
        CASE
            WHEN fact.flood_match_status = 'MATCHED' THEN level.flood_susceptibility_level
            WHEN fact.flood_match_status = 'UNMATCHED' AND fact.coordinate_status = 'VALID_PAIR'
                THEN 'OUTSIDE_MAPPED_ZONES'
            WHEN fact.flood_match_status = 'UNMATCHED' THEN 'NO_USABLE_POINT'
            ELSE fact.flood_match_status
        END AS flood_point_class
    FROM `03-gold`.fact_project_snapshot AS fact
    LEFT JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    LEFT JOIN `03-gold`.dim_flood_susceptibility AS level
        ON fact.flood_susceptibility_key = level.flood_susceptibility_key
    WHERE fact.is_current_snapshot = TRUE
        AND fact.region_key <> 0
),

class_totals AS (
    SELECT
        region_key,
        flood_point_class,
        COUNT(DISTINCT project_key) AS project_count,
        SUM(reported_budget_pesos) AS total_reported_budget_pesos,
        COUNT(DISTINCT CASE WHEN is_dpwh_flood_related THEN project_key END) AS flood_control_project_count,
        SUM(CASE WHEN is_dpwh_flood_related THEN reported_budget_pesos END) AS flood_control_budget_pesos,
        COUNT(DISTINCT CASE WHEN standardized_sector = 'Flood Control and Drainage' THEN project_key END)
            AS flood_control_single_sector_project_count,
        SUM(CASE WHEN standardized_sector = 'Flood Control and Drainage' THEN reported_budget_pesos END)
            AS flood_control_single_sector_budget_pesos,
        COUNT(DISTINCT CASE WHEN is_in_official_flood_list THEN project_key END) AS official_flood_list_project_count,
        SUM(CASE WHEN is_in_official_flood_list THEN reported_budget_pesos END) AS official_flood_list_budget_pesos,
        COUNT(DISTINCT CASE WHEN NOT is_dpwh_flood_related THEN project_key END) AS other_project_count,
        COUNT(DISTINCT CASE WHEN is_dpwh_flood_related IS NULL THEN project_key END) AS uncategorized_project_count
    FROM project_points
    GROUP BY ALL
),

latest_exposure_run AS (
    -- the newest gold exposure run, so older mgb snapshots never add up
    SELECT
        run_id,
        mgb_source_snapshot_id
    FROM `03-gold`.fact_region_flood_exposure
    QUALIFY ROW_NUMBER() OVER (ORDER BY source_load_ts DESC, run_id) = 1
),

level_exposure AS (
    SELECT
        exposure.region_key,
        level.flood_susceptibility_level AS flood_point_class,
        exposure.susceptible_area_sqkm,
        exposure.share_of_region_area_pct,
        exposure.level_area_additivity_status
    FROM `03-gold`.fact_region_flood_exposure AS exposure
    INNER JOIN latest_exposure_run AS latest
        ON exposure.run_id = latest.run_id
    INNER JOIN `03-gold`.dim_flood_susceptibility AS level
        ON exposure.flood_susceptibility_key = level.flood_susceptibility_key
    WHERE exposure.flood_susceptibility_key <> 0
),

region_exposure_status AS (
    -- one status per region. any level that is not additive blocks a summed area
    SELECT
        region_key,
        CASE
            WHEN COUNT_IF(level_area_additivity_status = 'NOT_EVALUATED') > 0 THEN 'NOT_EVALUATED'
            WHEN COUNT_IF(NOT (level_area_additivity_status <=> 'ADDITIVE_WITHIN_TOLERANCE')) > 0
                THEN 'NON_ADDITIVE_CROSS_LEVEL_OVERLAP'
            ELSE 'ADDITIVE_WITHIN_TOLERANCE'
        END AS exposure_area_status
    FROM level_exposure
    GROUP BY ALL
),

combined AS (
    SELECT
        region.region_key,
        region.region_name,
        region.region_short_name,
        region.source_coverage_status,
        point_class.flood_point_class,
        point_class.is_mgb_level,
        point_class.severity_rank,
        -- a region with no mapped dpwh project has unknown project figures, never 0
        CASE
            WHEN region.source_coverage_status <> 'NO_MAPPED_PROJECT_DATA' THEN COALESCE(totals.project_count, 0)
        END AS project_count,
        totals.total_reported_budget_pesos,
        CASE
            WHEN region.source_coverage_status <> 'NO_MAPPED_PROJECT_DATA'
                THEN COALESCE(totals.flood_control_project_count, 0)
        END AS flood_control_project_count,
        totals.flood_control_budget_pesos,
        CASE
            WHEN region.source_coverage_status <> 'NO_MAPPED_PROJECT_DATA'
                THEN COALESCE(totals.flood_control_single_sector_project_count, 0)
        END AS flood_control_single_sector_project_count,
        totals.flood_control_single_sector_budget_pesos,
        CASE
            WHEN region.source_coverage_status <> 'NO_MAPPED_PROJECT_DATA'
                THEN COALESCE(totals.official_flood_list_project_count, 0)
        END AS official_flood_list_project_count,
        totals.official_flood_list_budget_pesos,
        CASE
            WHEN region.source_coverage_status <> 'NO_MAPPED_PROJECT_DATA' THEN COALESCE(totals.other_project_count, 0)
        END AS other_project_count,
        CASE
            WHEN region.source_coverage_status <> 'NO_MAPPED_PROJECT_DATA'
                THEN COALESCE(totals.uncategorized_project_count, 0)
        END AS uncategorized_project_count,
        exposure.susceptible_area_sqkm,
        exposure.share_of_region_area_pct,
        COALESCE(status.exposure_area_status, 'NO_EXPOSURE_DATA') AS exposure_area_status,
        region.source_snapshot_id,
        latest.mgb_source_snapshot_id
    -- every region gets every class, so a class with no project still shows its area
    FROM regions AS region
    CROSS JOIN point_classes AS point_class
    LEFT JOIN class_totals AS totals
        ON region.region_key = totals.region_key
        AND point_class.flood_point_class = totals.flood_point_class
    LEFT JOIN level_exposure AS exposure
        ON region.region_key = exposure.region_key
        AND point_class.flood_point_class = exposure.flood_point_class
    LEFT JOIN region_exposure_status AS status
        ON region.region_key = status.region_key
    LEFT JOIN latest_exposure_run AS latest
        ON TRUE
)

SELECT
    region_key,
    region_name,
    region_short_name,
    source_coverage_status,
    flood_point_class,
    is_mgb_level,
    severity_rank,
    project_count,
    total_reported_budget_pesos,
    flood_control_project_count,
    flood_control_budget_pesos,
    flood_control_single_sector_project_count,
    flood_control_single_sector_budget_pesos,
    official_flood_list_project_count,
    official_flood_list_budget_pesos,
    other_project_count,
    uncategorized_project_count,
    ROUND(
        100.0 * flood_control_project_count
        / NULLIF(SUM(flood_control_project_count) OVER (PARTITION BY region_key), 0),
        4
    ) AS share_of_region_flood_control_projects_pct,
    ROUND(
        100.0 * flood_control_budget_pesos
        / NULLIF(SUM(flood_control_budget_pesos) OVER (PARTITION BY region_key), 0),
        4
    ) AS share_of_region_flood_control_budget_pct,
    susceptible_area_sqkm,
    share_of_region_area_pct,
    exposure_area_status,
    source_snapshot_id,
    mgb_source_snapshot_id
FROM combined
ORDER BY region_name, is_mgb_level DESC, severity_rank DESC NULLS LAST, flood_point_class;

## 4. Which projects need a closer look? (implementation monitoring)

**What:** build `analytics_implementation_monitoring`.

**Business question:** which recorded projects warrant further verification or follow-up?

**Grain:** one row per current DPWH project. Key `0` projects stay in, with the Gold name of key `0` at each level. The project carries its region and its province reporting area (D-57), so the dashboard can filter by either. The dashboard counts rows for the status distribution and the region, province, sector, office, and status drilldowns, and filters `needs_review = TRUE` for the exception table. Adding the reporting area does not change the grain, because Gold publishes exactly one reporting-area key per project.

**Why:** every field comes from Gold as published. The view adds plain review indicators that combine governed fields. Each one is a reason to look, not a finding:

| Indicator | Raised when |
| --- | --- |
| `review_active_zero_progress` | `status_group` is `ACTIVE`, the source status is **not** `For Procurement`, and `zero_progress_flag` is true |
| `review_active_long_running` | `status_group` is `ACTIVE` and Gold's `long_running_flag` is true ([D-32](../../docs/decisions.md), [D-49](../../docs/decisions.md)) |
| `review_active_progress_not_published` | `status_group` is `ACTIVE`, the source status is not `For Procurement`, and `physical_progress_pct` is `NULL` |
| `review_inactive_status` | `status_group` is `INACTIVE` (`Not Yet Started` or `Terminated`, [D-40](../../docs/decisions.md)) |
| `review_unresolved_status` | `status_group` is `NULL`. `status_mapping_status` says why. |

These states stay separate, and the view never merges them:

- **Zero reported progress** (`progress_reporting_status = 'ZERO'`) is a reported value of 0.
- **Progress not published** (`'NOT_PUBLISHED'`) means Gold has no valid value. Gold does not say whether it was missing, conflicting, or out of range.
- **For Procurement** projects are `ACTIVE` with 0% progress and no start date by design ([D-46](../../docs/decisions.md)). `is_for_procurement` marks them, and they never raise a progress indicator.
- **Inactive** is the approved D-40 group. **Long-running** is Gold's D-32 flag.

**Overdue is not built.** DPWH publishes a completion date, but the repo documents no trustworthy planned, target, or contractual completion date and no approved rule for one ([D-40](../../docs/decisions.md)). `source_completion_date` is shown as published and is never compared with a target.

**Interpretation rule:** a raised indicator means "verify this record". It does not mean stalled ([D-46](../../docs/decisions.md)), delayed, or overdue. Long-running does not mean delayed.

**Expected result:** one row per current Gold project. The rows add up to every current Gold project and budget.

In [0]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_implementation_monitoring (
    project_key COMMENT 'Project. Joins to dim_project.project_key. One row per current project.',
    contract_id COMMENT 'DPWH Contract ID exactly as published.',
    project_description COMMENT 'Project description from dim_project. NULL when the source rows disagree.',
    region_key COMMENT 'Region of the project. Joins to dim_region.region_key. 0 for Central Office, unmatched, and ambiguous projects.',
    region_name COMMENT 'Region name from dim_region, including the Gold name of key 0.',
    region_short_name COMMENT 'Short region label from dim_region.',
    region_match_status COMMENT 'Gold region match status, such as MATCHED, NON_GEOGRAPHIC for Central Office, or UNMATCHED.',
    reporting_area_key COMMENT 'Province reporting area of the project (D-57). Joins to dim_province_reporting_area.reporting_area_key. 0 for unresolved, ambiguous, and non-geographic projects.',
    reporting_area_name COMMENT 'Reporting-area name from dim_province_reporting_area, including the Gold name of key 0.',
    reporting_area_type COMMENT 'PROVINCE, INDEPENDENT_CITY, NCR, or the Gold type of key 0.',
    reporting_area_match_status COMMENT 'Gold reporting-area match status, such as MATCHED, MATCHED_WITH_CONFLICT, UNMAPPED, AMBIGUOUS, or NON_GEOGRAPHIC.',
    standardized_sector COMMENT 'Approved sector or Multi-sector (D-39). NULL when unmapped. infra_type_mapping_status says why.',
    infra_type_mapping_status COMMENT 'Why the sector is set or empty, copied from dim_project.',
    implementing_office COMMENT 'Implementing District Engineering Office from dim_project. NULL when Gold has no single resolved office.',
    infra_year COMMENT 'Infrastructure-year cohort from dim_project. A project cohort in the current snapshot. NULL when the source value is not a whole year.',
    source_status COMMENT 'DPWH status exactly as published, from dim_project_status. NULL when Gold has no status (status key 0).',
    standardized_status COMMENT 'Approved standardized status (D-40). NULL when the status is not mapped.',
    status_group COMMENT 'Approved status group FINISHED, ACTIVE, or INACTIVE (D-40). ACTIVE includes For Procurement. NULL when not mapped.',
    status_mapping_status COMMENT 'MAPPED, NO_SOURCE_STATUS when Gold has no status (key 0), or NO_APPROVED_STATUS_RULE when the source status has no approved mapping.',
    is_for_procurement COMMENT 'True when the DPWH source status is For Procurement. These projects have not started work and never raise a progress indicator (D-46).',
    start_date COMMENT 'Project start date from dim_date. NULL when missing.',
    source_completion_date COMMENT 'Completion date as published by DPWH, from dim_date. Not a target or contractual date. NULL when missing.',
    snapshot_date COMMENT 'File date of the DPWH snapshot. Gold measures long-running to this date, never to today.',
    physical_progress_pct COMMENT 'Reported physical progress from 0 to 100, from Gold. NULL when Gold has no valid value. Non-additive. Do not sum it.',
    progress_reporting_status COMMENT 'ZERO when physical_progress_pct is 0, ABOVE_ZERO when it is above 0, or NOT_PUBLISHED when it is NULL. NOT_PUBLISHED is not zero.',
    zero_progress_flag COMMENT 'Gold zero-progress flag. NULL when progress is unresolved. Includes For Procurement projects.',
    long_running_flag COMMENT 'Gold long-running flag (D-32, D-49). NULL when the span cannot be measured. Long-running does not mean delayed.',
    reported_budget_pesos COMMENT 'Reported project or contract budget in pesos. Not actual payment. NULL when missing.',
    review_active_zero_progress COMMENT 'True when the project is ACTIVE, not For Procurement, and zero_progress_flag is true. A reason to verify, not stalled.',
    review_active_long_running COMMENT 'True when the project is ACTIVE and long_running_flag is true. A reason to verify, not delayed.',
    review_active_progress_not_published COMMENT 'True when the project is ACTIVE, not For Procurement, and physical_progress_pct is NULL.',
    review_inactive_status COMMENT 'True when status_group is INACTIVE (Not Yet Started or Terminated, D-40).',
    review_unresolved_status COMMENT 'True when status_group is NULL. status_mapping_status says why.',
    review_reason_count COMMENT 'Number of review indicators that are true, from 0 to 5.',
    needs_review COMMENT 'True when at least one review indicator is true. Filter on it for the exception table.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).'
)
COMMENT 'Insight 4, implementation monitoring. One row per current DPWH project with region, province reporting area, governed status, progress, dates, long-running and zero-progress flags, and plain review indicators. Overdue is not built because no trustworthy target completion date exists. A review indicator is a reason to verify, not a finding of stalled or delayed work. Includes region key 0.'
AS
WITH current_snapshot AS (
    SELECT
        MAX(fact.source_snapshot_id) AS source_snapshot_id,
        MAX(snapshot_day.calendar_date) AS snapshot_date
    FROM `03-gold`.fact_project_snapshot AS fact
    LEFT JOIN `03-gold`.dim_date AS snapshot_day
        ON fact.snapshot_date_key = snapshot_day.date_key
    WHERE fact.is_current_snapshot = TRUE
),

monitored AS (
    SELECT
        fact.project_key,
        project.contract_id,
        project.project_description,
        fact.region_key,
        region.region_name,
        region.region_short_name,
        fact.region_match_status,
        fact.reporting_area_key,
        area.reporting_area_name,
        area.reporting_area_type,
        fact.reporting_area_match_status,
        project.standardized_sector,
        project.infra_type_mapping_status,
        project.implementing_office,
        project.infra_year,
        status.source_status,
        status.standardized_status,
        status.status_group,
        -- the same reason rule as view 5. status key 0 is no source status
        CASE
            WHEN fact.status_key = 0 THEN 'NO_SOURCE_STATUS'
            WHEN status.status_group IS NULL THEN 'NO_APPROVED_STATUS_RULE'
            ELSE 'MAPPED'
        END AS status_mapping_status,
        status.source_status <=> 'For Procurement' AS is_for_procurement,
        start_day.calendar_date AS start_date,
        completion_day.calendar_date AS source_completion_date,
        fact.physical_progress_pct,
        CASE
            WHEN fact.physical_progress_pct IS NULL THEN 'NOT_PUBLISHED'
            WHEN fact.physical_progress_pct = 0 THEN 'ZERO'
            ELSE 'ABOVE_ZERO'
        END AS progress_reporting_status,
        fact.zero_progress_flag,
        fact.long_running_flag,
        fact.reported_budget_pesos
    FROM `03-gold`.fact_project_snapshot AS fact
    LEFT JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    LEFT JOIN `03-gold`.dim_region AS region
        ON fact.region_key = region.region_key
    LEFT JOIN `03-gold`.dim_province_reporting_area AS area
        ON fact.reporting_area_key = area.reporting_area_key
    LEFT JOIN `03-gold`.dim_project_status AS status
        ON fact.status_key = status.status_key
    -- key 0 dates join to the unknown date member, so the date stays null
    LEFT JOIN `03-gold`.dim_date AS start_day
        ON fact.start_date_key = start_day.date_key
    LEFT JOIN `03-gold`.dim_date AS completion_day
        ON fact.source_completion_date_key = completion_day.date_key
    WHERE fact.is_current_snapshot = TRUE
),

indicators AS (
    -- each indicator is false unless its governed fields say so. the flags themselves stay as gold published them
    SELECT
        monitored.*,
        COALESCE(
            status_group = 'ACTIVE' AND NOT is_for_procurement AND zero_progress_flag, FALSE
        ) AS review_active_zero_progress,
        COALESCE(status_group = 'ACTIVE' AND long_running_flag, FALSE) AS review_active_long_running,
        COALESCE(
            status_group = 'ACTIVE' AND NOT is_for_procurement AND physical_progress_pct IS NULL, FALSE
        ) AS review_active_progress_not_published,
        COALESCE(status_group = 'INACTIVE', FALSE) AS review_inactive_status,
        status_group IS NULL AS review_unresolved_status
    FROM monitored
)

SELECT
    indicators.project_key,
    indicators.contract_id,
    indicators.project_description,
    indicators.region_key,
    indicators.region_name,
    indicators.region_short_name,
    indicators.region_match_status,
    indicators.reporting_area_key,
    indicators.reporting_area_name,
    indicators.reporting_area_type,
    indicators.reporting_area_match_status,
    indicators.standardized_sector,
    indicators.infra_type_mapping_status,
    indicators.implementing_office,
    indicators.infra_year,
    indicators.source_status,
    indicators.standardized_status,
    indicators.status_group,
    indicators.status_mapping_status,
    indicators.is_for_procurement,
    indicators.start_date,
    indicators.source_completion_date,
    current_snapshot.snapshot_date,
    indicators.physical_progress_pct,
    indicators.progress_reporting_status,
    indicators.zero_progress_flag,
    indicators.long_running_flag,
    indicators.reported_budget_pesos,
    indicators.review_active_zero_progress,
    indicators.review_active_long_running,
    indicators.review_active_progress_not_published,
    indicators.review_inactive_status,
    indicators.review_unresolved_status,
    CAST(indicators.review_active_zero_progress AS INT)
    + CAST(indicators.review_active_long_running AS INT)
    + CAST(indicators.review_active_progress_not_published AS INT)
    + CAST(indicators.review_inactive_status AS INT)
    + CAST(indicators.review_unresolved_status AS INT) AS review_reason_count,
    indicators.review_active_zero_progress
    OR indicators.review_active_long_running
    OR indicators.review_active_progress_not_published
    OR indicators.review_inactive_status
    OR indicators.review_unresolved_status AS needs_review,
    current_snapshot.source_snapshot_id
FROM indicators
CROSS JOIN current_snapshot;

## 5. Are we seeing projects or a connected infrastructure network? (infrastructure network)

**What:** build `analytics_infrastructure_network`.

**Business question:** what patterns emerge when we stop looking at each project in isolation? Where do projects cluster, where do corridor patterns or gaps appear, and how do project type, status, population, and flood context line up?

**Grain:** one row per current DPWH project, with its map point when Gold has one. Key `0` projects stay in.

**Why:**

- The view is lightweight. It joins the project fact to its dimensions once. It runs no pairwise distance, no graph or network computation, and no spatial intersection.
- `is_mappable` is true only for a valid coordinate pair. Projects without one stay in the view, so a map gap can be checked against them in view 6.
- `map_grid_latitude` and `map_grid_longitude` are the south-west corner of a 0.1-degree display cell (about 11 km). They are plain arithmetic on the point, for a density map when there are too many points to draw. They are not a spatial join and not a governed place.
- Flood context is the same flood point class as view 3. Population context is the newest primary population of the project's region, so it repeats on every project of a region. Never add it up.
- Status reason, sector, and cohort come from Gold exactly as in view 4.
- Province context is the D-57 reporting area from Gold, with its population. `region_province_hierarchy_status` shows when the project region and the parent region of its reporting area differ. Region is assigned from the reported region name first and province from the map point, so both are kept as Gold publishes them.

**Interpretation rule:** a project map shows spatial clusters, corridor patterns, concentration, and gaps in **recorded** projects. Gold holds no road or transport topology, so the map cannot show network connectivity. Do not call clustered projects a connected network. A gap can mean no project, or a project without a usable point.

**Expected result:** one row per current Gold project. The rows add up to every current Gold project and budget.

In [0]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_infrastructure_network (
    project_key COMMENT 'Project. Joins to dim_project.project_key. One row per current project.',
    contract_id COMMENT 'DPWH Contract ID exactly as published.',
    project_description COMMENT 'Project description from dim_project. NULL when the source rows disagree.',
    latitude COMMENT 'Project latitude in EPSG:4326 from Gold. NULL when the source has no agreed coordinate.',
    longitude COMMENT 'Project longitude in EPSG:4326 from Gold. NULL when the source has no agreed coordinate.',
    coordinate_status COMMENT 'VALID_PAIR, MISSING_PAIR, PARTIAL_PAIR, or OUT_OF_RANGE, from Gold.',
    is_mappable COMMENT 'True only when coordinate_status is VALID_PAIR. Map only these rows.',
    map_grid_latitude COMMENT 'South-west corner latitude of the 0.1-degree display cell, for a density map. Plain arithmetic, not a spatial join or a governed place. NULL when not mappable.',
    map_grid_longitude COMMENT 'South-west corner longitude of the 0.1-degree display cell, for a density map. Plain arithmetic, not a spatial join or a governed place. NULL when not mappable.',
    region_key COMMENT 'Region of the project. Joins to dim_region.region_key. 0 for Central Office, unmatched, and ambiguous projects.',
    region_name COMMENT 'Region name from dim_region, including the Gold name of key 0.',
    region_short_name COMMENT 'Short region label from dim_region.',
    region_match_status COMMENT 'Gold region match status, such as MATCHED, MATCHED_WITH_CONFLICT, or UNMATCHED.',
    reporting_area_key COMMENT 'Province reporting area of the project (D-57). Joins to dim_province_reporting_area.reporting_area_key. 0 for unresolved, ambiguous, and non-geographic projects.',
    reporting_area_name COMMENT 'Reporting-area name from dim_province_reporting_area, including the Gold name of key 0.',
    reporting_area_type COMMENT 'PROVINCE, INDEPENDENT_CITY, NCR, or the Gold type of key 0.',
    reporting_area_match_status COMMENT 'Gold reporting-area match status, such as MATCHED, MATCHED_WITH_CONFLICT, UNMAPPED, AMBIGUOUS, or NON_GEOGRAPHIC.',
    region_province_hierarchy_status COMMENT 'CONSISTENT when the project region is the parent region of its reporting area, CONFLICT when they differ, or NOT_EVALUATED when either key is 0. Both keys stay as Gold publishes them.',
    standardized_sector COMMENT 'Approved sector or Multi-sector (D-39). NULL when unmapped. infra_type_mapping_status says why.',
    infra_type_mapping_status COMMENT 'Why the sector is set or empty, copied from dim_project.',
    is_dpwh_flood_related COMMENT 'True when the project has a Flood Control and Drainage component (D-39). NULL when the category is unmapped.',
    is_in_official_flood_list COMMENT 'True when the Contract ID matches the official flood-control list.',
    infra_year COMMENT 'Infrastructure-year cohort from dim_project. A project cohort in the current snapshot. NULL when the source value is not a whole year.',
    status_group COMMENT 'Approved status group FINISHED, ACTIVE, or INACTIVE (D-40). NULL when not mapped.',
    status_mapping_status COMMENT 'MAPPED, NO_SOURCE_STATUS, or NO_APPROVED_STATUS_RULE, the same rule as analytics_implementation_monitoring.',
    reported_budget_pesos COMMENT 'Reported project or contract budget in pesos. Not actual payment. NULL when missing.',
    flood_point_class COMMENT 'Gold flood result at the project point, the same classes as analytics_flood_alignment. OUTSIDE_MAPPED_ZONES is not low risk.',
    flood_severity_rank COMMENT 'MGB severity rank when flood_point_class is a level. NULL otherwise.',
    region_population_count COMMENT 'Newest primary population of the project region from fact_region_population. Context only. It repeats on every project of the region, so never add it up. NULL for key 0.',
    population_reference_year COMMENT 'Census year of region_population_count.',
    reporting_area_population_count COMMENT 'Newest primary population of the project reporting area from fact_province_reporting_area_population. Context only. It repeats on every project of the area, so never add it up. NULL for key 0.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).'
)
COMMENT 'Insight 5, infrastructure network perspective. One row per current DPWH project with its map point, a 0.1-degree display cell, region and province reporting area, sector, status, cohort, reported budget, flood point class, and region and reporting-area population context. Shows spatial clusters, corridor patterns, and gaps in recorded projects. It cannot show network connectivity, because Gold has no road or transport topology. Includes region key 0.'
AS
WITH current_snapshot AS (
    SELECT MAX(source_snapshot_id) AS source_snapshot_id
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
),

primary_population AS (
    SELECT
        region_key,
        reference_year,
        population_count
    FROM `03-gold`.fact_region_population
    WHERE is_primary_source = TRUE
        AND region_key <> 0
    -- the newest primary census year only, so two years never add up
    QUALIFY reference_year = MAX(reference_year) OVER ()
),

area_population AS (
    SELECT
        reporting_area_key,
        population_count
    FROM `03-gold`.fact_province_reporting_area_population
    WHERE is_primary_source = TRUE
        AND reporting_area_key <> 0
    QUALIFY reference_year = MAX(reference_year) OVER ()
)

SELECT
    fact.project_key,
    project.contract_id,
    project.project_description,
    fact.latitude,
    fact.longitude,
    fact.coordinate_status,
    fact.coordinate_status <=> 'VALID_PAIR' AS is_mappable,
    CASE WHEN fact.coordinate_status <=> 'VALID_PAIR' THEN FLOOR(fact.latitude * 10) / 10 END AS map_grid_latitude,
    CASE WHEN fact.coordinate_status <=> 'VALID_PAIR' THEN FLOOR(fact.longitude * 10) / 10 END AS map_grid_longitude,
    fact.region_key,
    region.region_name,
    region.region_short_name,
    fact.region_match_status,
    fact.reporting_area_key,
    area.reporting_area_name,
    area.reporting_area_type,
    fact.reporting_area_match_status,
    -- the two levels come from different evidence. a difference is shown, never forced
    CASE
        WHEN fact.region_key = 0 OR fact.reporting_area_key = 0 THEN 'NOT_EVALUATED'
        WHEN area.region_key = fact.region_key THEN 'CONSISTENT'
        ELSE 'CONFLICT'
    END AS region_province_hierarchy_status,
    project.standardized_sector,
    project.infra_type_mapping_status,
    project.is_dpwh_flood_related,
    project.is_in_official_flood_list,
    project.infra_year,
    status.status_group,
    -- the same reason rule as view 4. status key 0 is no source status
    CASE
        WHEN fact.status_key = 0 THEN 'NO_SOURCE_STATUS'
        WHEN status.status_group IS NULL THEN 'NO_APPROVED_STATUS_RULE'
        ELSE 'MAPPED'
    END AS status_mapping_status,
    fact.reported_budget_pesos,
    -- the same flood point classes as view 3
    CASE
        WHEN fact.flood_match_status = 'MATCHED' THEN level.flood_susceptibility_level
        WHEN fact.flood_match_status = 'UNMATCHED' AND fact.coordinate_status = 'VALID_PAIR'
            THEN 'OUTSIDE_MAPPED_ZONES'
        WHEN fact.flood_match_status = 'UNMATCHED' THEN 'NO_USABLE_POINT'
        ELSE fact.flood_match_status
    END AS flood_point_class,
    CASE WHEN fact.flood_match_status = 'MATCHED' THEN level.severity_rank END AS flood_severity_rank,
    population.population_count AS region_population_count,
    population.reference_year AS population_reference_year,
    area_population.population_count AS reporting_area_population_count,
    current_snapshot.source_snapshot_id
FROM `03-gold`.fact_project_snapshot AS fact
LEFT JOIN `03-gold`.dim_project AS project
    ON fact.project_key = project.project_key
LEFT JOIN `03-gold`.dim_region AS region
    ON fact.region_key = region.region_key
LEFT JOIN `03-gold`.dim_project_status AS status
    ON fact.status_key = status.status_key
LEFT JOIN `03-gold`.dim_flood_susceptibility AS level
    ON fact.flood_susceptibility_key = level.flood_susceptibility_key
LEFT JOIN primary_population AS population
    ON fact.region_key = population.region_key
LEFT JOIN `03-gold`.dim_province_reporting_area AS area
    ON fact.reporting_area_key = area.reporting_area_key
LEFT JOIN area_population
    ON fact.reporting_area_key = area_population.reporting_area_key
CROSS JOIN current_snapshot
WHERE fact.is_current_snapshot = TRUE;

## 6. What can we see only after connecting the datasets? (integration coverage)

**What:** build `analytics_integration_coverage`.

**Business question:** how many project records connect to each governed source after the pipeline integrates them, and which records cannot be connected?

**Grain:** one row per integration link, link outcome, and region key. Region key `0` is included, so every current project appears **once in each link**. Never add rows across links. Adding the province links adds rows for new links only, so the grain is unchanged.

| `integration_link` | `link_outcome` comes from | `is_linked` is true when |
| --- | --- | --- |
| `PSGC_REGION` | Gold `region_match_status` | the project has an official region (`region_key <> 0`) |
| `PSGC_REPORTING_AREA` | Gold `reporting_area_match_status` | the project has a province reporting area (`reporting_area_key <> 0`, D-57) |
| `REGION_PROVINCE_HIERARCHY` | `CONSISTENT`, `CONFLICT`, or `NOT_EVALUATED`, the rule of view 5 | the project region is the parent region of its reporting area |
| `COORDINATES` | Gold `coordinate_status` | `VALID_PAIR` |
| `POPULATION_VIA_REGION` | `LINKED`, `REGION_WITHOUT_POPULATION`, or `NO_OFFICIAL_REGION` | the project's region has a primary population |
| `POPULATION_VIA_REPORTING_AREA` | `LINKED`, `REPORTING_AREA_WITHOUT_POPULATION`, or `NO_REPORTING_AREA` | the project's reporting area has a primary population |
| `FLOOD_SUSCEPTIBILITY` | `MATCHED`, `AMBIGUOUS`, `OUTSIDE_MAPPED_ZONES`, `NO_USABLE_POINT`, or `INVALID_SOURCE` | `MATCHED`, one MGB level |
| `PROJECT_CATEGORY` | Gold `infra_type_mapping_status` | `standardized_sector` is not `NULL` |
| `PROJECT_STATUS` | `status_mapping_status`, the rule of views 4 and 5 | `status_group` is not `NULL` |
| `REPORTED_BUDGET` | `REPORTED_ABOVE_ZERO`, `REPORTED_ZERO`, or `NOT_REPORTED` | the budget is not `NULL` |
| `OFFICIAL_FLOOD_LIST` | Gold `flood_list_match_status` | the Contract ID matches the official flood-control list |

**Why:** this view shows what Data Engineering adds. Each link is a source that Silver matched, standardized, or reconciled, and Gold published with lineage. The unlinked rows stay visible with their reason, so data-quality limits are part of the story.

- `pct_of_link_projects` and `pct_of_link_budget` divide by every current project and budget, so they add up across regions and outcomes to 100 within one link.
- `pct_of_region_projects` divides by the projects of the same region within the link.
- Budget sums only reported values. A missing budget is never counted as 0, so `reported_budget_pesos` is `NULL` when no project in the row has one.
- `OFFICIAL_FLOOD_LIST` is membership, not a quality gap. Most DPWH projects are not flood control, so a low linked share is expected.

**Interpretation rule:** "not linked" means the pipeline could not connect the record with the governed rules. It does not mean the project is wrong or missing in the field. A project outside mapped flood zones is not low risk.

**Expected result:** within each link, the rows add up to every current Gold project and budget. The linked and unlinked rows of each link reconcile with Gold.

In [0]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_integration_coverage (
    integration_link COMMENT 'Which governed connection this row measures: PSGC_REGION, PSGC_REPORTING_AREA, REGION_PROVINCE_HIERARCHY, COORDINATES, POPULATION_VIA_REGION, POPULATION_VIA_REPORTING_AREA, FLOOD_SUSCEPTIBILITY, PROJECT_CATEGORY, PROJECT_STATUS, REPORTED_BUDGET, or OFFICIAL_FLOOD_LIST. Every project appears once per link. Never add rows across links.',
    connected_source COMMENT 'Plain description of the source this link connects the DPWH project to.',
    link_outcome COMMENT 'Result of the link for the projects in this row, from the Gold status that the link names. NULL only when Gold publishes a NULL status.',
    is_linked COMMENT 'True when the projects in this row are connected for this link.',
    region_key COMMENT 'Region of the projects. Joins to dim_region.region_key. 0 for Central Office, unmatched, and ambiguous projects.',
    region_name COMMENT 'Region name from dim_region, including the Gold name of key 0.',
    project_count COMMENT 'Distinct current DPWH projects in the row. Adds up to every current project within one link.',
    projects_with_budget COMMENT 'Projects in the row with a reported budget that is not NULL.',
    reported_budget_pesos COMMENT 'Sum of reported budgets in the row. Not actual payment. NULL when no project in the row has a budget, never 0 for a missing value.',
    pct_of_link_projects COMMENT 'project_count as a percent of every current project. The rows of one link add up to 100.',
    pct_of_link_budget COMMENT 'reported_budget_pesos as a percent of every current reported budget. The rows of one link add up to 100.',
    pct_of_region_projects COMMENT 'project_count as a percent of the projects of the same region within the link.',
    source_snapshot_id COMMENT 'DPWH source snapshot behind every number in this view (D-29).'
)
COMMENT 'Insight 6, integration coverage. One row per integration link, link outcome, and region key with project count, reported budget, and shares, for eleven governed connections: PSGC region, province reporting area, region and province agreement, coordinates, population through region and through reporting area, MGB flood level, category, status, reported budget, and the official flood-control list. Every current project appears once in each link, including region key 0. Never add rows across links.'
AS
WITH current_snapshot AS (
    SELECT MAX(source_snapshot_id) AS source_snapshot_id
    FROM `03-gold`.fact_project_snapshot
    WHERE is_current_snapshot = TRUE
),

primary_population AS (
    SELECT DISTINCT region_key
    FROM `03-gold`.fact_region_population
    WHERE is_primary_source = TRUE
        AND region_key <> 0
    -- the newest primary census year only, the same rule as views 2 and 5
    QUALIFY reference_year = MAX(reference_year) OVER ()
),

area_population AS (
    SELECT DISTINCT reporting_area_key
    FROM `03-gold`.fact_province_reporting_area_population
    WHERE is_primary_source = TRUE
        AND reporting_area_key <> 0
        AND population_count IS NOT NULL
    QUALIFY reference_year = MAX(reference_year) OVER ()
),

projects AS (
    SELECT
        fact.project_key,
        fact.region_key,
        fact.reported_budget_pesos,
        fact.region_match_status,
        fact.reporting_area_key,
        fact.reporting_area_match_status,
        -- the same hierarchy rule as view 5
        CASE
            WHEN fact.region_key = 0 OR fact.reporting_area_key = 0 THEN 'NOT_EVALUATED'
            WHEN area.region_key = fact.region_key THEN 'CONSISTENT'
            ELSE 'CONFLICT'
        END AS region_province_hierarchy_status,
        area_population.reporting_area_key IS NOT NULL AS has_area_population,
        fact.coordinate_status,
        -- the same flood point classes as view 3, with every level collapsed to matched
        CASE
            WHEN fact.flood_match_status = 'UNMATCHED' AND fact.coordinate_status = 'VALID_PAIR'
                THEN 'OUTSIDE_MAPPED_ZONES'
            WHEN fact.flood_match_status = 'UNMATCHED' THEN 'NO_USABLE_POINT'
            ELSE fact.flood_match_status
        END AS flood_link_outcome,
        project.infra_type_mapping_status,
        project.standardized_sector,
        project.flood_list_match_status,
        project.is_in_official_flood_list,
        -- the same reason rule as views 4 and 5
        CASE
            WHEN fact.status_key = 0 THEN 'NO_SOURCE_STATUS'
            WHEN status.status_group IS NULL THEN 'NO_APPROVED_STATUS_RULE'
            ELSE 'MAPPED'
        END AS status_mapping_status,
        status.status_group,
        population.region_key IS NOT NULL AS has_population
    FROM `03-gold`.fact_project_snapshot AS fact
    LEFT JOIN `03-gold`.dim_project AS project
        ON fact.project_key = project.project_key
    LEFT JOIN `03-gold`.dim_project_status AS status
        ON fact.status_key = status.status_key
    LEFT JOIN primary_population AS population
        ON fact.region_key = population.region_key
    LEFT JOIN `03-gold`.dim_province_reporting_area AS area
        ON fact.reporting_area_key = area.reporting_area_key
    LEFT JOIN area_population
        ON fact.reporting_area_key = area_population.reporting_area_key
    WHERE fact.is_current_snapshot = TRUE
),

project_links AS (
    -- one pass over the projects. each project gets exactly one row per link
    SELECT
        projects.project_key,
        projects.region_key,
        projects.reported_budget_pesos,
        link.integration_link,
        link.link_outcome,
        link.is_linked
    FROM projects
    LATERAL VIEW INLINE(ARRAY(
        NAMED_STRUCT(
            'integration_link', 'PSGC_REGION',
            'link_outcome', projects.region_match_status,
            'is_linked', projects.region_key <> 0
        ),
        NAMED_STRUCT(
            'integration_link', 'PSGC_REPORTING_AREA',
            'link_outcome', projects.reporting_area_match_status,
            'is_linked', projects.reporting_area_key <> 0
        ),
        NAMED_STRUCT(
            'integration_link', 'REGION_PROVINCE_HIERARCHY',
            'link_outcome', projects.region_province_hierarchy_status,
            'is_linked', projects.region_province_hierarchy_status = 'CONSISTENT'
        ),
        NAMED_STRUCT(
            'integration_link', 'COORDINATES',
            'link_outcome', projects.coordinate_status,
            'is_linked', projects.coordinate_status <=> 'VALID_PAIR'
        ),
        NAMED_STRUCT(
            'integration_link', 'POPULATION_VIA_REGION',
            'link_outcome', CASE
                WHEN projects.region_key = 0 THEN 'NO_OFFICIAL_REGION'
                WHEN projects.has_population THEN 'LINKED'
                ELSE 'REGION_WITHOUT_POPULATION'
            END,
            'is_linked', projects.region_key <> 0 AND projects.has_population
        ),
        NAMED_STRUCT(
            'integration_link', 'POPULATION_VIA_REPORTING_AREA',
            'link_outcome', CASE
                WHEN projects.reporting_area_key = 0 THEN 'NO_REPORTING_AREA'
                WHEN projects.has_area_population THEN 'LINKED'
                ELSE 'REPORTING_AREA_WITHOUT_POPULATION'
            END,
            'is_linked', projects.reporting_area_key <> 0 AND projects.has_area_population
        ),
        NAMED_STRUCT(
            'integration_link', 'FLOOD_SUSCEPTIBILITY',
            'link_outcome', projects.flood_link_outcome,
            'is_linked', projects.flood_link_outcome <=> 'MATCHED'
        ),
        NAMED_STRUCT(
            'integration_link', 'PROJECT_CATEGORY',
            'link_outcome', projects.infra_type_mapping_status,
            'is_linked', projects.standardized_sector IS NOT NULL
        ),
        NAMED_STRUCT(
            'integration_link', 'PROJECT_STATUS',
            'link_outcome', projects.status_mapping_status,
            'is_linked', projects.status_group IS NOT NULL
        ),
        NAMED_STRUCT(
            'integration_link', 'REPORTED_BUDGET',
            'link_outcome', CASE
                WHEN projects.reported_budget_pesos IS NULL THEN 'NOT_REPORTED'
                WHEN projects.reported_budget_pesos = 0 THEN 'REPORTED_ZERO'
                ELSE 'REPORTED_ABOVE_ZERO'
            END,
            'is_linked', projects.reported_budget_pesos IS NOT NULL
        ),
        NAMED_STRUCT(
            'integration_link', 'OFFICIAL_FLOOD_LIST',
            'link_outcome', projects.flood_list_match_status,
            'is_linked', COALESCE(projects.is_in_official_flood_list, FALSE)
        )
    )) link
),

link_totals AS (
    SELECT
        integration_link,
        link_outcome,
        is_linked,
        region_key,
        COUNT(DISTINCT project_key) AS project_count,
        COUNT(DISTINCT CASE WHEN reported_budget_pesos IS NOT NULL THEN project_key END) AS projects_with_budget,
        SUM(reported_budget_pesos) AS reported_budget_pesos
    FROM project_links
    GROUP BY ALL
)

SELECT
    totals.integration_link,
    CASE totals.integration_link
        WHEN 'PSGC_REGION' THEN 'Official PSGC regions, through the Silver project-region mapping'
        WHEN 'PSGC_REPORTING_AREA' THEN 'PSGC provinces, independent cities, and NCR, through the Silver project-province mapping (D-57)'
        WHEN 'REGION_PROVINCE_HIERARCHY' THEN 'PSGC hierarchy between the project region and its reporting area'
        WHEN 'POPULATION_VIA_REPORTING_AREA' THEN 'PSA Census Table C population, through the reporting area'
        WHEN 'COORDINATES' THEN 'DPWH map points checked by Silver'
        WHEN 'POPULATION_VIA_REGION' THEN 'PSA Census Table C population, through the official region'
        WHEN 'FLOOD_SUSCEPTIBILITY' THEN 'DENR MGB flood-susceptibility polygons, through the Silver project-flood mapping'
        WHEN 'PROJECT_CATEGORY' THEN 'Approved DPWH componentCategories taxonomy (D-39)'
        WHEN 'PROJECT_STATUS' THEN 'Approved DPWH status mapping (D-40)'
        WHEN 'REPORTED_BUDGET' THEN 'DPWH reported project or contract budget'
        WHEN 'OFFICIAL_FLOOD_LIST' THEN 'BetterGov flood-control list, by exact Contract ID (D-04)'
    END AS connected_source,
    totals.link_outcome,
    totals.is_linked,
    totals.region_key,
    region.region_name,
    totals.project_count,
    totals.projects_with_budget,
    totals.reported_budget_pesos,
    ROUND(
        100.0 * totals.project_count
        / NULLIF(SUM(totals.project_count) OVER (PARTITION BY totals.integration_link), 0),
        4
    ) AS pct_of_link_projects,
    ROUND(
        100.0 * totals.reported_budget_pesos
        / NULLIF(SUM(totals.reported_budget_pesos) OVER (PARTITION BY totals.integration_link), 0),
        4
    ) AS pct_of_link_budget,
    ROUND(
        100.0 * totals.project_count
        / NULLIF(SUM(totals.project_count) OVER (PARTITION BY totals.integration_link, totals.region_key), 0),
        4
    ) AS pct_of_region_projects,
    current_snapshot.source_snapshot_id
FROM link_totals AS totals
LEFT JOIN `03-gold`.dim_region AS region
    ON totals.region_key = region.region_key
CROSS JOIN current_snapshot
ORDER BY totals.integration_link, totals.is_linked DESC, totals.project_count DESC, totals.region_key;

# Summary

Built the Analytics views from current-snapshot Gold, one per capstone insight, and dropped the retired views. The query below lists every view left in `05-analytics` with its row count. It should list exactly the Analytics views. Analytics validation runs separately in `04_validation/10_validation_analytics`.

In [0]:
%sql
SELECT 'analytics_investment_distribution' AS view_name, COUNT(*) AS row_count
FROM `05-analytics`.analytics_investment_distribution
UNION ALL
SELECT 'analytics_population_equity' AS view_name, COUNT(*) AS row_count
FROM `05-analytics`.analytics_population_equity
UNION ALL
SELECT 'analytics_flood_alignment' AS view_name, COUNT(*) AS row_count
FROM `05-analytics`.analytics_flood_alignment
UNION ALL
SELECT 'analytics_implementation_monitoring' AS view_name, COUNT(*) AS row_count
FROM `05-analytics`.analytics_implementation_monitoring
UNION ALL
SELECT 'analytics_infrastructure_network' AS view_name, COUNT(*) AS row_count
FROM `05-analytics`.analytics_infrastructure_network
UNION ALL
SELECT 'analytics_integration_coverage' AS view_name, COUNT(*) AS row_count
FROM `05-analytics`.analytics_integration_coverage;